In [ ]:
import pytz
import json
import math
import time
import schedule
import pandas as pd
import numpy as np
import yfinance as yf
from ib_async import *
from datetime import datetime
from pathlib import Path
from vix_config import *
from vix_ibkr import IBKR

util.startLoop()

In [4]:
class VIXContractManager:
    """
    Manages VIX futures contract resolution on IBKR.
    For directional strategy, we only need VX1 (front month).
    """
    def __init__(self, ibkr: IBKR):
        self.ibkr = ibkr

    def get_vix_futures_chain(self) -> list:
        """Fetch all available VIX futures contracts from IBKR, sorted by expiry."""
        vix_fut = Future(symbol='VIX', exchange='CFE', currency='USD')
        details = self.ibkr.ib.reqContractDetails(vix_fut)

        if not details:
            print("[ERROR] No VIX futures contracts found. Check market data subscriptions.")
            return []

        contracts = [d.contract for d in details]
        contracts.sort(key=lambda c: c.lastTradeDateOrContractMonth)
        return contracts

    def _get_active_contracts(self, min_count: int = 1) -> list:
        """Filter and return active standard monthly VIX futures, sorted by expiry."""
        chain = self.get_vix_futures_chain()
        today = datetime.now().strftime('%Y%m%d')
        active = [c for c in chain
                  if c.lastTradeDateOrContractMonth >= today
                  and c.tradingClass == 'VX']
        if len(active) < min_count:
            print(f"[ERROR] Need {min_count} active VIX futures, found {len(active)}")
        return active

    def get_front_month(self) -> Future:
        """Returns VX1 — the nearest active standard monthly VIX futures contract."""
        active = self._get_active_contracts(1)
        if not active:
            return None
        vx1 = active[0]
        self.ibkr.ib.qualifyContracts(vx1)
        print(f"[CONTRACT] VX1: {vx1.localSymbol} (exp: {vx1.lastTradeDateOrContractMonth})")
        return vx1

    def get_front_and_second_month(self) -> tuple:
        """Returns (VX1, VX2) for compatibility."""
        active = self._get_active_contracts(2)
        if len(active) < 2:
            return None, None
        vx1, vx2 = active[0], active[1]
        self.ibkr.ib.qualifyContracts(vx1)
        self.ibkr.ib.qualifyContracts(vx2)
        return vx1, vx2

    def get_vix_index(self) -> Index:
        """Returns the CBOE VIX spot index contract for signal generation."""
        vix = Index('VIX', 'CBOE', 'USD')
        self.ibkr.ib.qualifyContracts(vix)
        return vix

    def days_to_expiry(self, contract) -> int:
        """Calculate days until contract expiry."""
        exp = datetime.strptime(contract.lastTradeDateOrContractMonth, '%Y%m%d')
        return (exp - datetime.now()).days


In [ ]:
ib = IBKR()
ib.connect(np.random.randint(100000, 999999), isTWS=True)

Peer closed connection.


In [17]:
mgr = VIXContractManager(ib)
vx1 = mgr.get_front_month()
print(f"VX1: {vx1.localSymbol}, expire: {vx1.lastTradeDateOrContractMonth}")



[CONTRACT] VX1: VXJ6 (exp: 20260415)
VX1: VXJ6, expire: 20260415


In [12]:
class VIXDataEngine:
    """
    Fetches VIX spot (for signal), VX1 price (for execution), and maintains
    a rolling history for z-score computation.
    """

    def __init__(self, ibkr: IBKR, contract_mgr: VIXContractManager):
        self.ibkr = ibkr
        self.contract_mgr = contract_mgr
        self.vix_history = pd.DataFrame()  # columns: vix_spot, vx1, volume
        self._state_file = Path("vix_spot_history.csv")

    def load_state(self):
        """Load persisted VIX history from disk."""
        if self._state_file.exists() and self._state_file.stat().st_size > 0:
            try:
                self.vix_history = pd.read_csv(self._state_file, parse_dates=['date'], index_col='date')
                print(f"[DATA] Loaded {len(self.vix_history)} days of VIX history from disk.")
            except pd.errors.EmptyDataError:
                print("[DATA] History file is empty. Will build from scratch.")
        else:
            print("[DATA] No prior VIX history found. Will build from scratch.")

    def save_state(self):
        """Persist VIX history to disk."""
        if not self.vix_history.empty:
            self.vix_history.to_csv(self._state_file)
            print(f"[DATA] Saved {len(self.vix_history)} days of VIX history.")

    def fetch_historical_bars(self, contract, duration: str = DATA_DURATION,
                               bar_size: str = DATA_BAR_SIZE) -> pd.DataFrame:
        """Fetch historical OHLCV bars."""
        return self.ibkr.get_historical_prices(contract, duration=duration, bar_size=bar_size)

    def fetch_current_price(self, contract) -> float:
        """Fetch real-time (or delayed) price."""
        return self.ibkr.get_current_price(contract)

    def build_vix_history(self, vx1_contract) -> pd.DataFrame:
        """
        Build VIX spot + VX1 price history.
        VIX spot from yfinance (^VIX, accurate), VX1 from IBKR.
        """
        # Try yfinance first for real VIX spot history
        print("[DATA] Fetching VIX spot history from yfinance...")
        try:
            vix_yf = yf.download("^VIX", period="6mo", progress=False)
            if not vix_yf.empty:
                # Flatten multi-level columns if present
                if hasattr(vix_yf.columns, 'levels'):
                    vix_yf.columns = vix_yf.columns.get_level_values(0)
                vix_spot_series = vix_yf['Close'].dropna()
                vix_spot_series.index = pd.to_datetime(vix_spot_series.index).tz_localize(None)
                print(f"[DATA] Got {len(vix_spot_series)} days of VIX spot from yfinance.")
            else:
                vix_spot_series = pd.Series(dtype=float)
        except Exception as e:
            print(f"[WARN] yfinance VIX history failed: {e}")
            vix_spot_series = pd.Series(dtype=float)

        # Fetch VX1 history from IBKR
        print("[DATA] Fetching VX1 historical data from IBKR...")
        df_vx1 = self.fetch_historical_bars(vx1_contract)

        if df_vx1.empty and vix_spot_series.empty:
            print("[ERROR] Cannot build history — no data from any source.")
            return pd.DataFrame()

        # Build combined history
        if not vix_spot_series.empty and not df_vx1.empty:
            # Align on common dates
            common_idx = vix_spot_series.index.intersection(df_vx1.index)
            hist = pd.DataFrame({
                'vix_spot': vix_spot_series.reindex(common_idx),
                'vx1': df_vx1['close'].reindex(common_idx),
                'volume': df_vx1['volume'].reindex(common_idx, fill_value=0) if 'volume' in df_vx1.columns else 0
            }).dropna()
        elif not vix_spot_series.empty:
            # yfinance only — use VIX spot as VX1 proxy (better than nothing)
            hist = pd.DataFrame({
                'vix_spot': vix_spot_series,
                'vx1': vix_spot_series,
                'volume': 0
            }).dropna()
        else:
            # IBKR only — fallback to VX1 as VIX proxy (original behavior)
            hist = pd.DataFrame({
                'vix_spot': df_vx1['close'],
                'vx1': df_vx1['close'],
                'volume': df_vx1['volume'] if 'volume' in df_vx1.columns else 0
            }).dropna()

        hist.index.name = 'date'
        print(f"[DATA] Built VIX history: {len(hist)} days, "
              f"VIX spot range [{hist['vix_spot'].min():.2f}, {hist['vix_spot'].max():.2f}]")
        return hist

    def _fetch_vix_yfinance(self) -> float:
        """Fetch near-real-time VIX spot from Yahoo Finance."""
        try:
            vix = yf.Ticker("^VIX")
            price = vix.fast_info.get('lastPrice', float('nan'))
            if price and not math.isnan(price):
                print(f"[DATA] VIX spot from yfinance: {price:.2f}")
                return price
        except Exception as e:
            print(f"[WARN] yfinance VIX fetch failed: {e}")
        return float('nan')

    def fetch_vix_spot(self) -> float:
        """Fetch current VIX spot for signal generation and regime filtering."""
        price = self._fetch_vix_yfinance()
        if not math.isnan(price):
            return price

        print("[DATA] yfinance unavailable, falling back to IBKR.")
        vix_index = self.contract_mgr.get_vix_index()
        try:
            return self.ibkr.get_current_price(vix_index)
        except Exception:
            print("[ERROR] Could not get VIX spot from any source.")
            return float('nan')

    def fetch_latest_volume(self, contract) -> float:
        """Fetch latest daily bar volume."""
        try:
            df = self.ibkr.get_historical_prices(contract, duration='2 D', bar_size='1 day')
            if not df.empty and 'volume' in df.columns:
                return df['volume'].iloc[-1]
        except Exception as e:
            print(f"[WARN] Volume fetch failed: {e}")
        return 0.0

    def update_today(self, vix_spot: float, vx1_price: float, volume: float = 0.0):
        """Append today's data point to VIX history."""
        today = pd.Timestamp(datetime.now().date())

        new_row = pd.DataFrame({
            'vix_spot': [vix_spot],
            'vx1': [vx1_price],
            'volume': [volume],
        }, index=pd.DatetimeIndex([today], name='date'))

        if not self.vix_history.empty and today in self.vix_history.index:
            self.vix_history.loc[today] = new_row.iloc[0]
        else:
            self.vix_history = pd.concat([self.vix_history, new_row])

        max_rows = max(SPREAD_HISTORY_DAYS, Z_LOOKBACK * 3)
        if len(self.vix_history) > max_rows:
            self.vix_history = self.vix_history.iloc[-max_rows:]

        self.save_state()


In [20]:
price = ib.get_current_price(vx1)
print(f"VX1 price: {price}")


VX1 price: 24.65


In [21]:
engine = VIXDataEngine(ib, mgr)
hist = engine.build_vix_history(vx1)
print(f"history: {len(hist)} day")
hist

[DATA] Fetching VIX spot history from yfinance...


/var/folders/y7/ft6827m9701cqvkfl_918rwm0000gn/T/ipykernel_6245/2278358793.py:47: FutureWarning: YF.download() has changed argument auto_adjust default to True
  vix_yf = yf.download("^VIX", period="6mo", progress=False)


[DATA] Got 125 days of VIX spot from yfinance.
[DATA] Fetching VX1 historical data from IBKR...
[DATA] Built VIX history: 123 days, VIX spot range [13.47, 29.49]
history: 123 day


,vix_spot,vx1,volume
date,,,
2025-09-25,16.740000,21.75,208.0
2025-09-26,15.290000,21.35,330.0
2025-09-29,16.120001,21.45,271.0
2025-09-30,16.280001,21.45,290.0
2025-10-01,16.290001,21.45,460.0
...,...,...,...
2026-03-17,22.370001,22.70,74349.0
2026-03-18,25.090000,24.95,108516.0
2026-03-19,24.059999,23.85,113329.0


In [22]:
class VIXSignalEngine:
    """
    Computes rolling z-score of VIX spot price.
    Generates asymmetric entry signals:
      - SHORT when z > Z_ENTRY_SHORT (VIX high, contango helps)
      - LONG when z < -Z_ENTRY_LONG (VIX low, needs stronger signal)
    """

    def __init__(self, lookback: int = Z_LOOKBACK,
                 entry_short: float = Z_ENTRY_SHORT,
                 entry_long: float = Z_ENTRY_LONG):
        self.lookback = lookback
        self.entry_short = entry_short
        self.entry_long = entry_long

    def compute_zscore(self, series: pd.Series) -> pd.Series:
        """Rolling z-score: z = (value - rolling_mean) / rolling_std"""
        rolling_mean = series.rolling(window=self.lookback).mean()
        rolling_std = series.rolling(window=self.lookback).std().replace(0, np.nan)
        return (series - rolling_mean) / rolling_std

    def _check_slope_confirmation(self, vix_series: pd.Series, signal: str, verbose: bool) -> bool:
        """
        Check if VIX direction confirms mean-reversion.
        SHORT: VIX should be turning down (slope < 0)
        LONG:  VIX should be turning up (slope > 0)
        """
        if not SLOPE_CONFIRMATION:
            return True
        if len(vix_series) < SLOPE_LOOKBACK + 1:
            return True

        recent_slope = vix_series.iloc[-1] - vix_series.iloc[-1 - SLOPE_LOOKBACK]

        if verbose:
            print(f"Slope ({SLOPE_LOOKBACK}d): {recent_slope:+.4f}", end=" → ")

        if signal == 'LONG' and recent_slope < 0:
            if verbose:
                print("BLOCKED (VIX still falling)")
            return False
        elif signal == 'SHORT' and recent_slope > 0:
            if verbose:
                print("BLOCKED (VIX still rising)")
            return False

        if verbose:
            print("CONFIRMED")
        return True

    def _check_volume_confirmation(self, history: pd.DataFrame, verbose: bool) -> bool:
        """Check if current volume is above rolling average."""
        if not VOLUME_CONFIRMATION:
            return True
        if 'volume' not in history.columns:
            return True

        vol = history['volume']
        if len(vol) < VOLUME_LOOKBACK + 1:
            return True

        avg_vol = vol.rolling(VOLUME_LOOKBACK).mean().iloc[-1]
        current_vol = vol.iloc[-1]

        if avg_vol <= 0:
            return True

        ratio = current_vol / avg_vol

        if verbose:
            print(f"Volume: {current_vol:,.0f} vs avg {avg_vol:,.0f} "
                  f"(ratio: {ratio:.2f}, threshold: {VOLUME_MULTIPLIER})", end=" → ")

        if ratio < VOLUME_MULTIPLIER:
            if verbose:
                print("BLOCKED (below-average volume)")
            return False

        if verbose:
            print("CONFIRMED")
        return True

    def generate_signal(self, vix_history: pd.DataFrame, vix_spot: float,
                        vx1_price: float = None,
                        verbose: bool = VERBOSE) -> dict:
        """
        Generate directional trading signal based on VIX spot z-score + basis filter.

        Returns dict with:
            signal: 'SHORT', 'LONG', or 'NO_SIGNAL'
            z_score: current z-score value
            vix_spot: current VIX spot level
            regime_ok: whether regime allows this trade direction
        """
        if len(vix_history) < self.lookback + 1:
            if verbose:
                print(f"[SIGNAL] Insufficient data ({len(vix_history)} < {self.lookback + 1}).")
            return {'signal': 'NO_SIGNAL', 'z_score': 0.0, 'vix_spot': vix_spot,
                    'regime_ok': True, 'reason': 'insufficient_data'}

        vix_series = vix_history['vix_spot']
        z_series = self.compute_zscore(vix_series)
        current_z = z_series.iloc[-1]

        # Basis = VX1 - VIX Spot (positive = contango, negative = backwardation)
        basis = (vx1_price - vix_spot) if vx1_price is not None else None

        if verbose:
            print(f"\n--- VIX Signal Engine ---")
            print(f"VIX Spot: {vix_spot:.2f}")
            if vx1_price is not None:
                print(f"VX1 Price: {vx1_price:.2f} | Basis: {basis:+.2f} "
                      f"({'contango' if basis > 0 else 'backwardation'})")
            print(f"Z-Score: {current_z:.3f} (short: >{self.entry_short}, long: <-{self.entry_long})")

        # Asymmetric z-score entry signals
        signal = None
        if current_z > self.entry_short:
            signal = 'SHORT'
        elif current_z < -self.entry_long:
            signal = 'LONG'

        # Basis filter — short requires contango, long requires backwardation
        if signal is not None and basis is not None:
            if signal == 'SHORT' and basis <= 0:
                if verbose:
                    print(f"Basis BLOCKED: VX1 in backwardation (basis={basis:+.2f}), too risky to short")
                return {'signal': 'NO_SIGNAL', 'z_score': current_z, 'vix_spot': vix_spot,
                        'regime_ok': True, 'reason': f'basis_backwardation ({basis:+.2f})'}
            elif signal == 'LONG' and basis >= 0:
                if verbose:
                    print(f"Basis BLOCKED: VX1 in contango (basis={basis:+.2f}), contango hurts longs")
                return {'signal': 'NO_SIGNAL', 'z_score': current_z, 'vix_spot': vix_spot,
                        'regime_ok': True, 'reason': f'basis_contango ({basis:+.2f})'}

        # Regime filter — only blocks LONG entries (shorting high VIX is fine)
        if signal == 'LONG' and vix_spot >= VIX_REGIME_THRESHOLD:
            if verbose:
                print(f"Regime BLOCKED: VIX={vix_spot:.1f} >= {VIX_REGIME_THRESHOLD} (no longs)")
            return {'signal': 'NO_SIGNAL', 'z_score': current_z, 'vix_spot': vix_spot,
                    'regime_ok': False, 'reason': f'VIX={vix_spot:.1f} blocks longs'}

        # Confirmation filters
        if signal is not None:
            if not self._check_slope_confirmation(vix_series, signal, verbose):
                return {'signal': 'NO_SIGNAL', 'z_score': current_z, 'vix_spot': vix_spot,
                        'regime_ok': True, 'reason': 'slope_not_confirmed'}

            if not self._check_volume_confirmation(vix_history, verbose):
                return {'signal': 'NO_SIGNAL', 'z_score': current_z, 'vix_spot': vix_spot,
                        'regime_ok': True, 'reason': 'volume_not_confirmed'}

        final_signal = signal or 'NO_SIGNAL'
        if verbose:
            print(f"Signal: {final_signal}")
            print(f"------------------------\n")

        return {'signal': final_signal, 'z_score': current_z, 'vix_spot': vix_spot,
                'regime_ok': True,
                'reason': 'z_score_trigger' if signal else 'within_band'}


In [23]:
sig = VIXSignalEngine()
vix_spot = engine.fetch_vix_spot()

[DATA] VIX spot from yfinance: 26.15


In [24]:
signal = sig.generate_signal(hist, vix_spot, vx1_price=price, verbose=True)


--- VIX Signal Engine ---
VIX Spot: 26.15
VX1 Price: 24.65 | Basis: -1.50 (backwardation)
Z-Score: 0.595 (short: >0.75, long: <-1.0)
Signal: NO_SIGNAL
------------------------

